# Lab 06 · Schemas that let the model decline
**~25 minutes · costs about $0.02 · Domain 2 App Design (8.6%), Domain 6 Output Handling**

The heaviest objective on the exam includes schema design. One design choice
does most of the work: **nullable fields**. If a schema cannot express *"not
present"*, the model must invent something.

In [ ]:
import os, anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"            # verify against lab 00 output
CHEAP  = "claude-haiku-4-5"             # for high-volume steps
print("sdk", anthropic.__version__)

## A contract with a genuinely missing field

In [ ]:
CONTRACT = """MASTER SERVICES AGREEMENT
Vendor: Northwind Logistics Ltd.
Effective: 3 March 2026.
This agreement renews automatically and continues until terminated by either
party with 90 days written notice. Payment terms are net 45."""

import json
def extract(schema, label):
    r = client.messages.create(model=MODEL, max_tokens=400,
        system=("Extract fields from the contract. Return ONLY JSON matching this schema:\n"
                + json.dumps(schema)),
        messages=[{"role":"user","content":CONTRACT}])
    txt = r.content[0].text.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    print(label, "->", txt.strip())
    return txt

## Version A · no way to say "unknown"

There is **no end date** in this contract; it runs until terminated. Watch what
comes back.

In [ ]:
A = {"type":"object","properties":{
        "vendor":{"type":"string"},
        "contract_end_date":{"type":"string","description":"ISO date"}},
     "required":["vendor","contract_end_date"]}
extract(A, "A (not nullable)");

## Version B · nullable, with the reason spelled out

Same model, same contract. The only change is that declining is now
representable.

In [ ]:
B = {"type":"object","properties":{
        "vendor":{"type":"string"},
        "contract_end_date":{"type":["string","null"],
            "description":"ISO date, or null if the contract has no fixed end date"}},
     "required":["vendor","contract_end_date"]}
extract(B, "B (nullable)");

Run both a few times. A tends to fabricate a plausible date; B tends to return
`null`. Marking a field *required* makes fabrication **more** likely, not less
— it removes the escape hatch while insisting on a value.

## Validation is not the same as correctness

In [ ]:
import json
def validate(raw):
    try:
        d = json.loads(raw)
    except json.JSONDecodeError as e:
        return f"INVALID JSON: {e}"
    if "vendor" not in d:
        return "SCHEMA FAIL: missing vendor"
    if d.get("contract_end_date") and not str(d["contract_end_date"])[:4].isdigit():
        return "SEMANTIC FAIL: end date is not a date"
    return f"passes both: {d}"

print(validate('{"vendor":"Northwind Logistics Ltd.","contract_end_date":null}'))
print(validate('{"vendor":"Northwind Logistics Ltd.","contract_end_date":"2031-03-03"}'))
print("   ^ schema-valid and completely fabricated. Validation cannot catch this.")

## The truncation trap

Force `max_tokens` mid-JSON. The instinct is to send it back for repair. Do not.

In [ ]:
r = client.messages.create(model=MODEL, max_tokens=20,
        system="Return ONLY a JSON object with 12 fields describing the contract.",
        messages=[{"role":"user","content":CONTRACT}])
print("stop_reason:", r.stop_reason)
print("payload:", repr(r.content[0].text))
print()
if r.stop_reason == "max_tokens":
    print("Detect this BEFORE parsing. Raise the budget or decompose the task.")
    print("A repair loop asks the model to invent data that was never generated.")

---
### Checkpoint
- Why does a required non-nullable field increase fabrication?
- Difference between schema-valid and semantically correct?
- What do you check before `json.loads`?